# Preencher datas e escolher filial

In [ ]:
# Data de faturamento (a partir de)
data_inicial_saida_entrega = ""

# Data a ser exibida no excel do protocolo de entrega
data_no_protocolo = ""

# Escolher filial ao trocar chaves da requisição. Altera nome do arquivo excel gerado
# filial = "distribuidora"
# filial = "comercio"

# Imports

In [15]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

In [16]:
# Acumulador dos df_nfs de cada execução (uma por filial). Só inicializa se ainda não existir
# no kernel, para não perder o resultado da filial já rodada ao reexecutar com outra filial.
if "lista_df_nfs" not in globals():
    lista_df_nfs = []

# Request API

In [17]:
url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env e escolhe credenciais por filial
load_dotenv()
# Procura variáveis específicas por filial, com fallback para variáveis genéricas
if filial.lower() == "distribuidora":
    app_key = os.getenv("APP_KEY_DISTRIBUIDORA") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_DISTRIBUIDORA") or os.getenv("app_secret") or os.getenv("APP_SECRET")
else:
    app_key = os.getenv("APP_KEY_COMERCIO") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_COMERCIO") or os.getenv("app_secret") or os.getenv("APP_SECRET")
print(f"Usando credenciais para filial: {filial}")

payload = {
    "call": "ListarNF",
    "app_key": app_key,
    "app_secret": app_secret,
    "param": [{
  "pagina": 1,
  "registros_por_pagina": 100,
  "ordenar_por": "CODIGO",
  "dSaiEntInicial": data_inicial_saida_entrega,
  "tpNF": 1
}]
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"Erro na requisição da página 1")
else:
    data = response.json()

# Extrai os dados das notas fiscais do json e armazena em uma lista de tuplas (cnpj_cpf, numero_nf)
nfs = data.get("nfCadastro", [])
todas_nfs = []
for nf in nfs:
    numero_nf = nf.get('ide', {}).get('nNF').lstrip('0')
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    todas_nfs.append((cnpj_cpf, numero_nf))

print(todas_nfs)

Usando credenciais para filial: comercio
[('02.318.956/0020-24', '9850'), ('34.035.902/0003-47', '9851'), ('34.035.902/0002-66', '9852'), ('10.254.717/0008-90', '9853'), ('02.318.956/0005-95', '9854'), ('02.318.956/0003-23', '9855'), ('02.318.956/0008-38', '9856'), ('34.035.902/0006-90', '9857'), ('10.254.717/0002-02', '9858'), ('02.318.956/0007-57', '9859'), ('02.318.956/0004-04', '9860'), ('02.318.956/0020-24', '9861'), ('12.365.759/0002-38', '9862'), ('02.318.956/0001-61', '9863'), ('12.365.759/0003-19', '9864'), ('12.365.759/0001-57', '9865'), ('04.879.925/0001-05', '9866'), ('45.236.954/0001-36', '9867'), ('45.236.954/0002-17', '9868'), ('10.254.717/0004-66', '9869'), ('12.365.759/0004-08', '9870'), ('34.035.902/0003-47', '9871'), ('10.254.717/0005-47', '9872'), ('34.035.902/0002-66', '9873'), ('10.254.717/0004-66', '9874'), ('02.318.956/0005-95', '9875'), ('02.318.956/0003-23', '9876'), ('02.318.956/0013-03', '9877'), ('02.318.956/0008-38', '9878'), ('02.318.956/0006-76', '9879')

# Dataframe de Produtos das NFs

In [18]:
# Monta uma linha por produto de cada NF, com os dados solicitados
registros = []
for nf in nfs:
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    nnf = nf.get('ide', {}).get('nNF', '').lstrip('0')
    nid_transp = nf.get('compl', {}).get('nIdTransp')

    for item in nf.get('det', []):
        prod = item.get('prod', {})
        registros.append({
            'cnpj_cpf': cnpj_cpf,
            'xProd': prod.get('xProd'),
            'qCom': prod.get('qCom'),
            'uCom': prod.get('uCom'),
            'nnf': nnf,
            'nIdTransp': nid_transp,
            'filial': filial,
        })

df_nfs = pd.DataFrame(registros)
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial
0,02.318.956/0020-24,BATATA ESCOVADA FL (KG),50,KG,9850,8598091217,comercio
1,34.035.902/0003-47,BATATA ASTERIX (KG),75,KG,9851,8603320478,comercio
2,34.035.902/0002-66,BATATA ASTERIX (KG),50,KG,9852,8603320478,comercio
3,10.254.717/0008-90,BATATA ASTERIX (KG),75,KG,9853,8603320478,comercio
4,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9854,8603320478,comercio
...,...,...,...,...,...,...,...
112,12.365.759/0001-57,BATATA ESCOVADA NV (KG),50,KG,9946,8598091687,comercio
113,04.879.925/0001-05,BATATA ASTERIX (KG),75,KG,9947,8598091217,comercio
114,04.879.925/0001-05,BATATA ESCOVADA FL (KG),50,KG,9947,8598091217,comercio
115,45.236.954/0001-36,BATATA ESCOVADA FL (KG),75,KG,9948,12330332636,comercio


# Nome Fantasia no Dataframe

In [19]:
# Adiciona a coluna de nome fantasia no df_nfs a partir de cnpj_nomefantasia.csv
df_nomes = pd.read_csv("cnpj_nomefantasia.csv")
cnpj_to_fantasia = dict(zip(df_nomes["CNPJ"], df_nomes["nome_fantasia"]))
df_nfs["nome_fantasia"] = df_nfs["cnpj_cpf"].map(cnpj_to_fantasia)
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,02.318.956/0020-24,BATATA ESCOVADA FL (KG),50,KG,9850,8598091217,comercio,PIRAJÁ CIDADE SP
1,34.035.902/0003-47,BATATA ASTERIX (KG),75,KG,9851,8603320478,comercio,ASTOR SP
2,34.035.902/0002-66,BATATA ASTERIX (KG),50,KG,9852,8603320478,comercio,ASTOR JK
3,10.254.717/0008-90,BATATA ASTERIX (KG),75,KG,9853,8603320478,comercio,ICI ALPHAVILLE
4,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9854,8603320478,comercio,PIRAJÁ VILLA LOBOS
...,...,...,...,...,...,...,...,...
112,12.365.759/0001-57,BATATA ESCOVADA NV (KG),50,KG,9946,8598091687,comercio,LC6
113,04.879.925/0001-05,BATATA ASTERIX (KG),75,KG,9947,8598091217,comercio,ICI BISTRÔ
114,04.879.925/0001-05,BATATA ESCOVADA FL (KG),50,KG,9947,8598091217,comercio,ICI BISTRÔ
115,45.236.954/0001-36,BATATA ESCOVADA FL (KG),75,KG,9948,12330332636,comercio,IXE COTOXÓ


# Transportador no Dataframe

In [20]:
# Carrega o mapeamento nIdTransp -> transportador a partir de nIDTransp.txt (um json por linha)
mapa_transportadores = []
with open("nIDTransp.txt", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if linha:
            mapa_transportadores.append(json.loads(linha))

nid_to_transportador = {m["nIdTransp"]: m["transportador"] for m in mapa_transportadores}

# Substitui nIdTransp pelo nome do transportador correspondente
df_nfs["nIdTransp"] = df_nfs["nIdTransp"].astype(str).map(nid_to_transportador).fillna(df_nfs["nIdTransp"])
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,02.318.956/0020-24,BATATA ESCOVADA FL (KG),50,KG,9850,Jaylton,comercio,PIRAJÁ CIDADE SP
1,34.035.902/0003-47,BATATA ASTERIX (KG),75,KG,9851,Wesley,comercio,ASTOR SP
2,34.035.902/0002-66,BATATA ASTERIX (KG),50,KG,9852,Wesley,comercio,ASTOR JK
3,10.254.717/0008-90,BATATA ASTERIX (KG),75,KG,9853,Wesley,comercio,ICI ALPHAVILLE
4,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9854,Wesley,comercio,PIRAJÁ VILLA LOBOS
...,...,...,...,...,...,...,...,...
112,12.365.759/0001-57,BATATA ESCOVADA NV (KG),50,KG,9946,Jaylton,comercio,LC6
113,04.879.925/0001-05,BATATA ASTERIX (KG),75,KG,9947,Jaylton,comercio,ICI BISTRÔ
114,04.879.925/0001-05,BATATA ESCOVADA FL (KG),50,KG,9947,Jaylton,comercio,ICI BISTRÔ
115,45.236.954/0001-36,BATATA ESCOVADA FL (KG),75,KG,9948,Luan,comercio,IXE COTOXÓ


# Consolidar Dataframes das Filiais

In [21]:
# Substitui no acumulador o resultado anterior desta filial (evita duplicar ao reexecutar a mesma filial)
# e junta com os resultados das demais filiais já rodadas neste kernel
lista_df_nfs = [df for df in lista_df_nfs if not (df["filial"] == filial).all()]
lista_df_nfs.append(df_nfs)

df_nfs_consolidado = pd.concat(lista_df_nfs, ignore_index=True)
df_nfs_consolidado

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,62.452.992/0001-45,BATATA ASTERIX (KG),400,KG,42576,Wesley,distribuidora,ZDELI ITAIM
1,62.251.536/0001-37,BATATA ASTERIX (KG),350,KG,42577,Jaylton,distribuidora,ZDELI REBOUÇAS
2,58.558.570/0001-81,BATATA ASTERIX (KG),250,KG,42578,Jaylton,distribuidora,ZDELI HADDOCK LOBO
3,58.558.570/0001-81,BATATA LAVADA (KG),50,KG,42578,Jaylton,distribuidora,ZDELI HADDOCK LOBO
4,18.786.642/0004-76,BATATA ESCOVADA FL (KG),250,KG,42579,Jaylton,distribuidora,ZDELI IAB - REPUBLICA
...,...,...,...,...,...,...,...,...
180,12.365.759/0001-57,BATATA ESCOVADA NV (KG),50,KG,9946,Jaylton,comercio,LC6
181,04.879.925/0001-05,BATATA ASTERIX (KG),75,KG,9947,Jaylton,comercio,ICI BISTRÔ
182,04.879.925/0001-05,BATATA ESCOVADA FL (KG),50,KG,9947,Jaylton,comercio,ICI BISTRÔ
183,45.236.954/0001-36,BATATA ESCOVADA FL (KG),75,KG,9948,Luan,comercio,IXE COTOXÓ


# Sacos de Batata

In [22]:
# Cria sacos_batata: para produtos com "batata" no nome, qCom convertido em sacos de 25kg
eh_batata = df_nfs_consolidado["xProd"].str.contains("batata", case=False, na=False)
df_nfs_consolidado["sacos_batata"] = None
df_nfs_consolidado.loc[eh_batata, "sacos_batata"] = df_nfs_consolidado.loc[eh_batata, "qCom"] / 25
df_nfs_consolidado

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia,sacos_batata
0,62.452.992/0001-45,BATATA ASTERIX (KG),400,KG,42576,Wesley,distribuidora,ZDELI ITAIM,16.0
1,62.251.536/0001-37,BATATA ASTERIX (KG),350,KG,42577,Jaylton,distribuidora,ZDELI REBOUÇAS,14.0
2,58.558.570/0001-81,BATATA ASTERIX (KG),250,KG,42578,Jaylton,distribuidora,ZDELI HADDOCK LOBO,10.0
3,58.558.570/0001-81,BATATA LAVADA (KG),50,KG,42578,Jaylton,distribuidora,ZDELI HADDOCK LOBO,2.0
4,18.786.642/0004-76,BATATA ESCOVADA FL (KG),250,KG,42579,Jaylton,distribuidora,ZDELI IAB - REPUBLICA,10.0
...,...,...,...,...,...,...,...,...,...
180,12.365.759/0001-57,BATATA ESCOVADA NV (KG),50,KG,9946,Jaylton,comercio,LC6,2.0
181,04.879.925/0001-05,BATATA ASTERIX (KG),75,KG,9947,Jaylton,comercio,ICI BISTRÔ,3.0
182,04.879.925/0001-05,BATATA ESCOVADA FL (KG),50,KG,9947,Jaylton,comercio,ICI BISTRÔ,2.0
183,45.236.954/0001-36,BATATA ESCOVADA FL (KG),75,KG,9948,Luan,comercio,IXE COTOXÓ,3.0


# Sacos de Batata por Transportador

In [23]:
# Soma de sacos_batata por xProd e nIdTransp (apenas produtos de batata), ordena por nIdTransp e exporta
df_sacos_por_transportador = (
    df_nfs_consolidado[df_nfs_consolidado["sacos_batata"].notna()]
    .groupby(["nIdTransp", "xProd"], as_index=False)["sacos_batata"]
    .sum()
    .sort_values("nIdTransp")
)

df_sacos_por_transportador.to_excel("output/sacos_por_transportador.xlsx", index=False)
df_sacos_por_transportador

,nIdTransp,xProd,sacos_batata
0,Jaylton,BATATA ASTERIX (KG),305.0
1,Jaylton,BATATA ESCOVADA FL (KG),119.0
2,Jaylton,BATATA ESCOVADA NV (KG),71.0
3,Jaylton,BATATA LAVADA (KG),3.0
4,Luan,BATATA ASTERIX (KG),22.0
5,Luan,BATATA BOLINHA 1- (KG),0.6
6,Luan,BATATA ESCOVADA FL (KG),39.0
7,Wesley,BATATA ASTERIX (KG),163.0
8,Wesley,BATATA ESCOVADA FL (KG),48.0


# Exportar Protocolo Consolidado

In [24]:
# Exporta df_nfs_consolidado apenas com as colunas solicitadas, na ordem pedida, ordenado por nIdTransp
colunas_exportacao = ["nIdTransp", "nome_fantasia", "xProd", "sacos_batata"]
df_nfs_consolidado[colunas_exportacao].sort_values("nIdTransp").to_excel("output/protocolo_nfs_consolidado.xlsx", index=False)

# Algoritmo Nome Fantasia

In [25]:
#importa "clientes_concatenados_atualizada.csv" para variável df_concatenated
df_concatenated = pd.read_csv("clientes_concatenados_atualizada.csv")

#cria cnpj_col para verificar qual coluna de CNPJ existe
cnpj_col = 'cnpj_cpf' if 'cnpj_cpf' in df_concatenated.columns else 'CNPJ'

# substitui os CNPJs de todas_nfs pelos nomes fantasia usando o df_concatenated atualizado
nome_fantasia_col = 'nome_fantasia' if 'nome_fantasia' in df_concatenated.columns else 'Nome Fantasia'
cnpj_to_nome = dict(zip(df_concatenated[cnpj_col], df_concatenated[nome_fantasia_col]))
todas_nfs = [(cnpj_to_nome.get(cnpj, cnpj), num) for cnpj, num in todas_nfs]

print(f"Substituição com df_concatenated atualizado concluída. Total de NFs: {len(todas_nfs)}")
print(todas_nfs)


Substituição com df_concatenated atualizado concluída. Total de NFs: 100
[('PIRA CIDADE SP', '9850'), ('ASTOR SP', '9851'), ('ASTOR JK', '9852'), ('ICI ALPHAVILLE', '9853'), ('PIRA VL LOBOS', '9854'), ('PIRA ALPHAVILLE', '9855'), ('PIRA ELDORADO', '9856'), ('ASTOR VL NOVA CONCEICAO', '9857'), ('ICI BELA CINTRA', '9858'), ('PIRA PAULISTA', '9859'), ('PIRA MORUMBI', '9860'), ('PIRA CIDADE SP', '9861'), ('LC2', '9862'), ('PIRA PINHEIRO', '9863'), ('LC3', '9864'), ('LC6', '9865'), ('ICI BISTRÔ', '9866'), ('IXE COTOXÓ', '9867'), ('IXE CARAÍBAS', '9868'), ('ICI VL LOBOS', '9869'), ('LC4', '9870'), ('ASTOR SP', '9871'), ('ICI JK', '9872'), ('ASTOR JK', '9873'), ('ICI VL LOBOS', '9874'), ('PIRA VL LOBOS', '9875'), ('PIRA ALPHAVILLE', '9876'), ('PIRA TAMBORE', '9877'), ('PIRA ELDORADO', '9878'), ('ORIGINAL', '9879'), ('ASTOR VL NOVA CONCEICAO', '9880'), ('ASTOR CETENCO', '9881'), ('ASTOR OF', '9882'), ('ICI BELA CINTRA', '9883'), ('PIRA PAULISTA', '9884'), ('PIRA MORUMBI', '9885'), ('PIRA CIDAD

# Algoritmo Planilha Protocolo

In [26]:
# Configuração de blocos
blocos_por_planilha = 5
linhas_por_bloco = 13
linha_inicio = 7

workbook_index = 1
block_index = 0

wb = load_workbook("protocolo_entrega_alterado.xlsx")
ws = wb.active

for i in range(0, len(todas_nfs), 2):
    # Inicia um novo workbook a cada blocos_por_planilha blocos
    if block_index >= blocos_por_planilha:
        file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"

        celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
        for celula in celulas:
            nova_img = Image('logo_v2.png')
            nova_img.width = 110
            nova_img.height = 95
            ws.add_image(nova_img, celula)

        wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")

        print(f"Salvou {file_name} com {block_index} blocos")

        workbook_index += 1
        block_index = 0
        wb = load_workbook("protocolo_entrega_alterado.xlsx")
        ws = wb.active

    lote = todas_nfs[i:i+2]
    print(f"Processando lote {block_index+1} da planilha {workbook_index}: {lote}")

    row_base = linha_inicio + block_index * linhas_por_bloco
    print(f"Base row para este bloco: {row_base}")

    if len(lote) == 2:
        cnpj1, num1 = lote[0]
        cnpj2, num2 = lote[1]

        # Escreve CNPJs em pares na mesma linha
        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base, column=11, value=cnpj2)  # Coluna K

        # Escreve data_no_protocolo em pares na linha seguinte
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo)   # Coluna C
        ws.cell(row=row_base + 1, column=12, value=data_no_protocolo)  # Coluna K

        # Escreve números em pares na linha seguinte
        ws.cell(row=row_base + 3, column=4, value=num1)  # Coluna C
        ws.cell(row=row_base + 3, column=12, value=num2) # Coluna K

        print(f"Bloco {block_index+1} planilha {workbook_index}: row {row_base}/{row_base+3} -> {cnpj1},{cnpj2}/{num1},{num2}")

    elif len(lote) == 1:
        cnpj1, num1 = lote[0]

        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo) # Coluna C
        ws.cell(row=row_base + 3, column=3, value=num1) # Coluna C (seguindo o padrão de par)
        
       

        print(f"Bloco {block_index+1} planilha {workbook_index} (solitário): row {row_base}/{row_base+3} -> {cnpj1}/{num1}")

    else:
        continue

    block_index += 1

# Salva o último workbook em aberto
if block_index > 0:
    file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"
    
    celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
    for celula in celulas:
        nova_img = Image('logo_v2.png')
        nova_img.width = 110
        nova_img.height = 95
        ws.add_image(nova_img, celula)

    wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")
    print(f"Salvou {file_name} com {block_index} blocos")

Processando lote 1 da planilha 1: [('PIRA CIDADE SP', '9850'), ('ASTOR SP', '9851')]
Base row para este bloco: 7
Bloco 1 planilha 1: row 7/10 -> PIRA CIDADE SP,ASTOR SP/9850,9851
Processando lote 2 da planilha 1: [('ASTOR JK', '9852'), ('ICI ALPHAVILLE', '9853')]
Base row para este bloco: 20
Bloco 2 planilha 1: row 20/23 -> ASTOR JK,ICI ALPHAVILLE/9852,9853
Processando lote 3 da planilha 1: [('PIRA VL LOBOS', '9854'), ('PIRA ALPHAVILLE', '9855')]
Base row para este bloco: 33
Bloco 3 planilha 1: row 33/36 -> PIRA VL LOBOS,PIRA ALPHAVILLE/9854,9855
Processando lote 4 da planilha 1: [('PIRA ELDORADO', '9856'), ('ASTOR VL NOVA CONCEICAO', '9857')]
Base row para este bloco: 46
Bloco 4 planilha 1: row 46/49 -> PIRA ELDORADO,ASTOR VL NOVA CONCEICAO/9856,9857
Processando lote 5 da planilha 1: [('ICI BELA CINTRA', '9858'), ('PIRA PAULISTA', '9859')]
Base row para este bloco: 59
Bloco 5 planilha 1: row 59/62 -> ICI BELA CINTRA,PIRA PAULISTA/9858,9859
Salvou comercio_protocolo_entrega_parte_1.xls